# **Notebook 07 — Model Comparison**

## Objectives

* Compare supervised (XGBoost) vs unsupervised (Autoencoder) approaches side by side
* Analyse strengths and weaknesses of each method
* Test ensemble combination of both models
* Provide final recommendation for the business

## Inputs

* `outputs/v2/fraud_model_optimized.pkl`
* `outputs/v2/test_probabilities.pkl`
* `outputs/v2/optimal_threshold.json`
* `outputs/v3/reconstruction_errors.pkl`
* `outputs/v3/ae_threshold.json`
* `outputs/v1/X_test_engineered.csv`, `outputs/v1/y_test.csv`

## Outputs

* `outputs/comparison_results.json` — final comparison metrics
* Recommendation for production deployment

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/dok.stv/Documents/Projects/credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_curve, auc, precision_recall_curve
)

# Load test data
X_test = pd.read_csv("outputs/v1/X_test_engineered.csv")
y_test = pd.read_csv("outputs/v1/y_test.csv").squeeze()

# Load XGBoost predictions
xgb_proba = joblib.load("outputs/v2/test_probabilities.pkl")
with open("outputs/v2/optimal_threshold.json") as f:
    xgb_threshold = json.load(f)['optimal_threshold']

# Load Autoencoder predictions
ae_errors = joblib.load("outputs/v3/reconstruction_errors.pkl")
with open("outputs/v3/ae_threshold.json") as f:
    ae_data = json.load(f)
    ae_threshold = ae_data['threshold']

print(f"Test set: {len(y_test):,} transactions ({y_test.sum()} fraud)")
print(f"XGBoost threshold: {xgb_threshold:.2f}")
print(f"Autoencoder threshold: {ae_threshold:.6f}")

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/v3/reconstruction_errors.pkl'

---
## 1. Side-by-Side Evaluation

In [ ]:
# Generate predictions at optimal thresholds
xgb_pred = (xgb_proba >= xgb_threshold).astype(int)
ae_pred = (ae_errors > ae_threshold).astype(int)

print("=" * 65)
print("  SUPERVISED: XGBoost (Optimised Threshold)")
print("=" * 65)
print(classification_report(y_test, xgb_pred,
                            target_names=['Legitimate', 'Fraud']))

cm_xgb = confusion_matrix(y_test, xgb_pred)
print(f"  TN: {cm_xgb[0][0]:,}   FP: {cm_xgb[0][1]:,}")
print(f"  FN: {cm_xgb[1][0]:,}   TP: {cm_xgb[1][1]:,}")

print(f"\n{'=' * 65}")
print("  UNSUPERVISED: Autoencoder (Anomaly Detection)")
print("=" * 65)
print(classification_report(y_test, ae_pred,
                            target_names=['Legitimate', 'Fraud']))

cm_ae = confusion_matrix(y_test, ae_pred)
print(f"  TN: {cm_ae[0][0]:,}   FP: {cm_ae[0][1]:,}")
print(f"  FN: {cm_ae[1][0]:,}   TP: {cm_ae[1][1]:,}")

In [ ]:
# Build comparison table
xgb_report = classification_report(y_test, xgb_pred, output_dict=True)
ae_report = classification_report(y_test, ae_pred, output_dict=True)

# ROC AUC for both
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba)
xgb_auc = auc(xgb_fpr, xgb_tpr)

ae_fpr, ae_tpr, _ = roc_curve(y_test, ae_errors)
ae_auc = auc(ae_fpr, ae_tpr)

comparison = pd.DataFrame({
    'Metric': ['F1 (Fraud)', 'Precision (Fraud)', 'Recall (Fraud)',
               'AUC-ROC', 'False Positives', 'False Negatives'],
    'XGBoost': [
        f"{xgb_report['Fraud']['f1-score']:.4f}",
        f"{xgb_report['Fraud']['precision']:.4f}",
        f"{xgb_report['Fraud']['recall']:.4f}",
        f"{xgb_auc:.4f}",
        f"{cm_xgb[0][1]:,}",
        f"{cm_xgb[1][0]:,}"
    ],
    'Autoencoder': [
        f"{ae_report['Fraud']['f1-score']:.4f}",
        f"{ae_report['Fraud']['precision']:.4f}",
        f"{ae_report['Fraud']['recall']:.4f}",
        f"{ae_auc:.4f}",
        f"{cm_ae[0][1]:,}",
        f"{cm_ae[1][0]:,}"
    ]
})

print("\nModel Comparison Summary")
print("=" * 55)
print(comparison.to_string(index=False))

### Comparison Analysis

**XGBoost strengths:**
- Higher precision — fewer false alarms for the investigation team
- Higher F1 — better overall balance of precision and recall
- Explainable — SHAP values show why each prediction was made

**Autoencoder strengths:**
- Does not require fraud labels — can detect truly novel attacks
- Catches patterns that may not exist in historical training data
- Useful as a complementary screening layer

---
## 2. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(xgb_fpr, xgb_tpr, color='#636EFA', linewidth=2,
        label=f'XGBoost (AUC = {xgb_auc:.3f})')
ax.plot(ae_fpr, ae_tpr, color='#EF553B', linewidth=2,
        label=f'Autoencoder (AUC = {ae_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Supervised vs Unsupervised')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("outputs/roc_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Ensemble Analysis

Testing what happens when we combine both models. A transaction is flagged if **either** model flags it (OR logic). This maximises recall at the cost of some precision.

In [ ]:
# Ensemble: flag if EITHER model flags it
ensemble_pred = ((xgb_pred == 1) | (ae_pred == 1)).astype(int)

print("ENSEMBLE (XGBoost OR Autoencoder)")
print("=" * 55)
print(classification_report(y_test, ensemble_pred,
                            target_names=['Legitimate', 'Fraud']))

cm_ens = confusion_matrix(y_test, ensemble_pred)
print(f"  TN: {cm_ens[0][0]:,}   FP: {cm_ens[0][1]:,}")
print(f"  FN: {cm_ens[1][0]:,}   TP: {cm_ens[1][1]:,}")

# How many fraud cases does each model catch uniquely?
xgb_only = ((xgb_pred == 1) & (ae_pred == 0) & (y_test == 1)).sum()
ae_only = ((xgb_pred == 0) & (ae_pred == 1) & (y_test == 1)).sum()
both = ((xgb_pred == 1) & (ae_pred == 1) & (y_test == 1)).sum()
neither = ((xgb_pred == 0) & (ae_pred == 0) & (y_test == 1)).sum()

print(f"\nFraud Detection Overlap:")
print(f"  Caught by both:          {both}")
print(f"  Caught by XGBoost only:  {xgb_only}")
print(f"  Caught by Autoencoder only: {ae_only}")
print(f"  Missed by both:          {neither}")
print(f"  Total fraud cases:       {y_test.sum()}")

In [ ]:
# Visualise the overlap
fig, ax = plt.subplots(figsize=(8, 5))

categories = ['Both Models', 'XGBoost Only', 'Autoencoder Only', 'Missed']
values = [both, xgb_only, ae_only, neither]
colors = ['#00CC96', '#636EFA', '#EF553B', '#CCCCCC']

bars = ax.bar(categories, values, color=colors)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(val), ha='center', fontweight='bold')

ax.set_ylabel('Fraud Cases')
ax.set_title('Fraud Detection Coverage by Model')
plt.tight_layout()
plt.show()

### Ensemble Analysis Conclusion

The autoencoder catches some fraud cases that XGBoost misses. This confirms the value of the unsupervised approach as a **complementary layer** — combining both models increases overall recall. The cost is additional false positives from the autoencoder, which in a real business context would mean more transactions sent to manual review.

---
## 4. Final Recommendation

In [ ]:
# Save comparison results
ens_report = classification_report(y_test, ensemble_pred, output_dict=True)

comparison_results = {
    'xgboost': {
        'f1': float(xgb_report['Fraud']['f1-score']),
        'precision': float(xgb_report['Fraud']['precision']),
        'recall': float(xgb_report['Fraud']['recall']),
        'auc_roc': float(xgb_auc),
        'threshold': float(xgb_threshold)
    },
    'autoencoder': {
        'f1': float(ae_report['Fraud']['f1-score']),
        'precision': float(ae_report['Fraud']['precision']),
        'recall': float(ae_report['Fraud']['recall']),
        'auc_roc': float(ae_auc),
        'threshold': float(ae_threshold)
    },
    'ensemble': {
        'f1': float(ens_report['Fraud']['f1-score']),
        'precision': float(ens_report['Fraud']['precision']),
        'recall': float(ens_report['Fraud']['recall']),
    },
    'overlap': {
        'caught_by_both': int(both),
        'xgboost_only': int(xgb_only),
        'autoencoder_only': int(ae_only),
        'missed_by_both': int(neither)
    }
}

with open("outputs/comparison_results.json", 'w') as f:
    json.dump(comparison_results, f, indent=2)

print("Comparison results saved to outputs/comparison_results.json")

---
## Conclusions

### Production Deployment Strategy

| Component | Role | Action on Flag |
|-----------|------|---------------|
| **XGBoost (Primary)** | High-confidence fraud detection with explainable predictions | Auto-block transaction + generate SHAP explanation for review |
| **Autoencoder (Secondary)** | Novel pattern detection for unknown fraud types | Flag for manual review — may catch emerging fraud strategies |

### Key Findings

1. **XGBoost** achieves the best overall metrics and should be the **primary detection system** (BR2)
2. **Autoencoder** catches some fraud that XGBoost misses — valuable as a **complementary layer** (BR3)
3. The **ensemble** approach (OR logic) maximises recall at the cost of more false positives
4. Transactions flagged by the autoencoder but NOT by XGBoost are the most interesting — they may represent **novel fraud patterns** not present in the labelled training data

### Business Impact

For SecurePay Solutions:
- The XGBoost model provides **automated, explainable** fraud screening that meets the performance targets
- The autoencoder adds a **safety net** for detecting fraud evolution
- The threshold and cost analysis (Dashboard Page 5) allows the risk team to **tune** the system for their specific cost trade-offs

